# LLM Evaluation
Compare two grounded-answer prompts using Groq (OpenAI-compatible API) as an LLM judge. Set  before running.

In [ ]:
import os
import pandas as pd
from openai import OpenAI

MODEL = os.getenv('LLM_MODEL', 'llama-3.1-8b-instant')
client = OpenAI(
    api_key=os.getenv('GROQ_API_KEY', ''),
    base_url='https://api.groq.com/openai/v1',
)

In [ ]:
test_cases = [
    {
        'query': 'What is deep learning?',
        'context': 'Deep learning is part of a broader family of machine learning methods based on artificial neural networks with representation learning.'
    },
    {
        'query': 'What is backpropagation?',
        'context': 'Backpropagation is a method used in artificial neural networks to calculate a gradient needed to update network weights.'
    }
]

In [ ]:
prompt_templates = {
    'Concise assistant': '''Answer the question based ONLY on the context. If unsupported, say you do not have enough information.\nContext: {context}\nQuestion: {query}\nAnswer:''',
    'ML educator': '''You are a machine-learning educator. Give a clear, concise explanation using ONLY the context. If unsupported, say exactly: I don't have enough information to answer that based on my knowledge base.\nContext: {context}\nQuestion: {query}\nAnswer:''',
}

In [ ]:
def complete(prompt):
    response = client.chat.completions.create(
        model=MODEL, messages=[{'role': 'user', 'content': prompt}], temperature=0.0
    )
    return response.choices[0].message.content.strip()

def evaluate_answer(query, context, answer):
    judge_prompt = f'''Rate this answer from 1 to 5 for relevance and faithfulness to the context. Return one integer only.\nQuestion: {query}\nContext: {context}\nAnswer: {answer}'''
    raw_score = complete(judge_prompt)
    try:
        return int(raw_score)
    except ValueError:
        return 0

In [ ]:
results = []
for name, template in prompt_templates.items():
    scores = []
    for case in test_cases:
        answer = complete(template.format(**case))
        score = evaluate_answer(case['query'], case['context'], answer)
        scores.append(score)
    results.append({'Prompt': name, 'Average score': sum(scores) / len(scores)})

pd.DataFrame(results).sort_values('Average score', ascending=False)